# KD240 上的 MNIST MLP —— 250 MHz 版本

和 `../mnist_kd240.bit` 用的是同一份 HLS IP，差別只在重新實作時把 PL 時脈約束
從 100 MHz 改成 250 MHz。繞線後 WNS 是 +0.010 ns，所以 250 MHz 是**達標**，
不是超頻硬撐出來的。

本板實測：

| PL 時脈 | 預測錯誤張數 | 吞吐量 |
|---|---|---|
| 250 MHz（簽核值） | 0 | 168,698 fps |
| 300 MHz | 0 | 200,333 fps |
| 375 MHz | 0 | **248,122 fps** |
| 500 MHz | 8,977 | 已超過極限 |

和 v1 notebook 有兩個差異：7.84 MB 的資料從 host 複製進 DMA buffer 只做**一次**，
而且在計時區間之外（v1 每次迭代都重做一遍，等於把 host memcpy 藏進了 FPGA 的
數字裡）；另外 buffer 的 flush/invalidate 是明確呼叫的，不是碰運氣。

In [ ]:
from pynq import Overlay, allocate, ps
import numpy as np
import time

N_IMAGES, PIXELS = 10000, 784

overlay = Overlay('mnist_kd240.bit')
regs = overlay.MultilayerPerceptron_0.register_map

# PL 時脈會跨 session 保留 —— 載入 bitstream 並不會把它重設。
# 所以這裡明確設定，而不是假設它就等於設計頻率。
ps.Clocks.fclk0_mhz = 249.997
print('PL  %.3f MHz' % ps.Clocks.fclk0_mhz)
print('CPU %.2f MHz' % ps.Clocks.cpu_mhz)

In [ ]:
in_buf = allocate(shape=(N_IMAGES * PIXELS,), dtype=np.int8)
out_buf = allocate(shape=(N_IMAGES,), dtype=np.int8)
regs.im_1.im = in_buf.device_address
regs.out_r_1.out_r = out_buf.device_address

x_test = (np.load('../x_test.npy') // 32).astype(np.int8).reshape(N_IMAGES, PIXELS)
y_test = np.load('../y_test.npy').astype(np.uint8)
golden = np.frombuffer(open('golden_pred_i8.bin', 'rb').read(), dtype=np.int8)

in_buf[:] = x_test.reshape(-1)   # 前置準備，不算推論時間，只做一次
in_buf.flush()
print('loaded', x_test.shape)

In [ ]:
def mnist_hw():
    out_buf[:] = 0
    out_buf.flush()
    regs.CTRL.AP_START = 1
    while regs.CTRL.AP_DONE == 0:
        pass
    out_buf.invalidate()
    return np.array(out_buf)

res = mnist_hw()
print('mismatches vs golden :', int((res != golden).sum()))
print('accuracy             : %.4f' % (res == y_test).mean())

`mismatches vs golden` 必須是 **0**。光看準確率抓不到時脈設錯 ——
一個已經超過時序極限的設計，準確率照樣有 ~0.97，但個別預測是錯的。

In [ ]:
t = %timeit -n 1 -r 10 -o mnist_hw()

mhz = ps.Clocks.fclk0_mhz
print('per image  : %.3f us' % (t.average / N_IMAGES * 1e6))
print('throughput : {:,.0f} fps'.format(N_IMAGES / t.average))
print('cycles/img : %.0f at %.0f MHz' % (t.average / N_IMAGES * mhz * 1e6, mhz))

每張約 1482 cycles，合成估計則是 1411。多出來的部分是每個 batch 固定約 2.7 ms
的快取維護加上 Python 輪詢 `AP_DONE` 的時間，不是加速器本身的時間。

In [ ]:
# 時脈掃描。PS 的 IOPLL 只能產生 1500/N MHz，所以只有這幾階可以選。
for target in [249.997, 299.997, 374.996]:
    ps.Clocks.fclk0_mhz = target
    mnist_hw()                                    # 暖機，結果丟棄
    t0 = time.time()
    r = mnist_hw()
    dt = time.time() - t0
    print('{:7.1f} MHz  wrong={:<6d} {:,.0f} fps'.format(
        ps.Clocks.fclk0_mhz, int((r != golden).sum()), N_IMAGES / dt))

ps.Clocks.fclk0_mhz = 249.997
print('restored to %.3f MHz' % ps.Clocks.fclk0_mhz)

500 MHz 故意不放進上面的清單：它會在 10,000 張裡產生 8,977 張錯誤預測。
375 MHz 是這顆晶片在室溫下的上限 —— 結果是 bit-exact 的，但那是超頻，
換溫度或換一批晶片都不保證。250 MHz 才是 Vivado 簽核的數字。

## 板子上的硬體 vs 軟體比較

上面的檢查是拿 host 端算好的參考答案來比對。這個 cell 直接在 KD240 上，
用 Vitis AI 匯出的權重重建同一個量化網路，所以整個比較不依賴 host 端的工具鏈。

光看準確率是很弱的檢查 —— 兩個實作可以都拿 0.9764，卻在不同的圖片上出錯。
`hls == py` 數的是兩者**完全一致**的張數，那個數字必須是 10000。

In [ ]:
import glob

W = [np.loadtxt(f) for f in sorted(glob.glob('../VitisAI/dump_results/dump_results_weights/quant_dense_*_kernel.txt'))]
B = [np.loadtxt(f) for f in sorted(glob.glob('../VitisAI/dump_results/dump_results_weights/quant_dense_*_bias.txt'))]
layers, scales = [784, 128, 256, 10], [512, 256, 256]
W = [W[i].reshape(layers[i], layers[i + 1]) for i in range(3)]

def mnist_sw(images):
    """同一個網路的 numpy 版本，逐張處理 —— 依照文章的寫法。"""
    out = []
    for i in range(len(images)):
        d = images[i]
        for j in range(3):
            d = (d @ W[j] + B[j]) // scales[j]
            if j != 2:
                d = d * (d > 0)
        out.append(np.argmax(d))
    return np.array(out)

ps.Clocks.fclk0_mhz = 249.997

t0 = time.time(); res_hls = mnist_hw(); t_hls = time.time() - t0
t0 = time.time(); res_py = mnist_sw(x_test); t_py = time.time() - t0

In [ ]:
print('acc hls  %.4f' % (res_hls == y_test).mean())
print('acc py   %.4f' % (res_py == y_test).mean())
print('hls == py: {}/{} images'.format(int((res_hls == res_py).sum()), N_IMAGES))
print()
print('hls fps  {:>10,.1f}'.format(N_IMAGES / t_hls))
print('py  fps  {:>10,.1f}'.format(N_IMAGES / t_py))
print('speedup  {:>10.1f}x'.format(t_py / t_hls))

`hls == py` 必須是 10000。如果兩邊準確率一樣、但這個數字不是 10000，
代表硬體在某些圖片上錯、某些圖片上僥倖對了 —— 這正是時脈超過極限時的樣子。

上面的 speedup 是拿文章那個逐張的 Python 迴圈當基準，而那是個很慢的基準。
在這塊板子的 ARM 核心上實測：

| 實作方式 | 吞吐量 |
|---|---|
| 逐張 Python 迴圈（文章的寫法） | 916 fps |
| 向量化 numpy，同樣的算式 | 7,458 fps |
| FPGA @ 250 MHz | 168,698 fps |
| FPGA @ 375 MHz | 248,122 fps |

所以誠實的數字是 **對比原始 Python 271x**、**對比向量化 numpy 33x**。
跟 CPU 比較時要引用 33x 這個數字。

（向量化版本必須維持 float64 才對得上：numpy 的整數 matmul 沒有 BLAS 路徑，
硬改成 int64 反而會比迴圈慢 2 倍。）